In [3]:
import os
import random
import numpy as np
import torch
import torch.nn.functional as F
import librosa
from torch.utils.data import Dataset, DataLoader
import torchaudio.transforms as T

In [4]:
import os
import numpy as np
import librosa
import random
from tqdm import tqdm

# Settings
SR = 22050
GLOBAL_DURATION = 30
PROCESSED_DIR = "/kaggle/working/processed_mels"
ROOT_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

def pre_process():
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    genres = sorted([d for d in os.listdir(ROOT_DIR) if os.path.isdir(os.path.join(ROOT_DIR, d))])
    
    for genre in genres:
        genre_in = os.path.join(ROOT_DIR, genre)
        genre_out = os.path.join(PROCESSED_DIR, genre)
        os.makedirs(genre_out, exist_ok=True)
        
        songs = [s for s in os.listdir(genre_in) if os.path.isdir(os.path.join(genre_in, s))]
        for song in tqdm(songs, desc=f"Processing {genre}"):
            song_out = os.path.join(genre_out, song)
            os.makedirs(song_out, exist_ok=True)
            
            for stem in ["drums.wav", "bass.wav", "vocals.wav", "other.wav"]:
                in_file = os.path.join(genre_in, song, stem)
                out_file = os.path.join(song_out, stem.replace(".wav", ".npy"))
                
                if os.path.exists(out_file): continue
                
                # Load and convert
                y, _ = librosa.load(in_file, sr=SR, duration=GLOBAL_DURATION)
                if len(y) < SR * GLOBAL_DURATION:
                    y = np.pad(y, (0, SR * GLOBAL_DURATION - len(y)))
                
                # We save Linear Mel Power (not DB) to allow mixing/scaling later
                mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=128, n_fft=2048, hop_length=512)
                np.save(out_file, mel.astype(np.float32))

# Run the pre-processor
pre_process()

Processing rock: 100%|██████████| 100/100 [01:07<00:00,  1.48it/s]


In [9]:
import os
import numpy as np
import librosa
from tqdm import tqdm

# Basic settings
sample_rate = 22050
num_mels = 128
hop_length = 512
fft_size = 2048

input_noise_folder = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio"
output_noise_folder = "/kaggle/working/processed_noise_mels"

def preprocess_noise():

    # Create output folder if it doesn't exist
    if not os.path.exists(output_noise_folder):
        os.makedirs(output_noise_folder)

    # Get all .wav files
    noise_files = []
    for file in os.listdir(input_noise_folder):
        if file.endswith(".wav"):
            noise_files.append(file)

    print(f"Total noise files found: {len(noise_files)}")

    # Process each noise file
    for file in tqdm(noise_files):

        input_path = os.path.join(input_noise_folder, file)
        output_path = os.path.join(output_noise_folder, file.replace(".wav", ".npy"))

        # Skip if already processed
        if os.path.exists(output_path):
            continue

        try:
            # Load full audio file
            audio, sr = librosa.load(input_path, sr=sample_rate)

            # Convert to Mel Spectrogram
            mel_spec = librosa.feature.melspectrogram(
                y=audio,
                sr=sample_rate,
                n_mels=num_mels,
                n_fft=fft_size,
                hop_length=hop_length
            )

            # Save as numpy file
            np.save(output_path, mel_spec.astype(np.float32))

        except Exception as error:
            print(f"Error while processing {file}")
            print(error)

    print("Noise preprocessing complete!")

# Run the function
preprocess_noise()

Total noise files found: 2000


100%|██████████| 2000/2000 [00:00<00:00, 144676.07it/s]

Noise preprocessing complete!


In [10]:
import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torchaudio.transforms as T
import librosa

def get_stratified_split(root_dir, val_ratio=0.15):
    train_songs = {}
    val_songs = {}
    genres = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])

    for genre in genres:
        genre_path = os.path.join(root_dir, genre)
        # List all song subdirectories
        all_songs = sorted([s for s in os.listdir(genre_path) if os.path.isdir(os.path.join(genre_path, s))])
        random.shuffle(all_songs)
        
        split_idx = int(len(all_songs) * val_ratio)
        val_songs[genre] = all_songs[:split_idx]
        train_songs[genre] = all_songs[split_idx:]
        
        print(f"Genre {genre:10} | Train: {len(train_songs[genre]):3} | Val: {len(val_songs[genre]):2}")
    
    return train_songs, val_songs, genres

In [11]:
class ASTRobustDataset(Dataset):
    def __init__(self, processed_dir, noise_dir, song_dict, genres, size=30000, is_val=False):
        self.processed_dir = processed_dir
        self.noise_files = [os.path.join(noise_dir, f) for f in os.listdir(noise_dir) if f.endswith(".npy")]
        self.song_dict = song_dict
        self.genres = genres
        self.genre_to_idx = {g: i for i, g in enumerate(genres)}
        self.size = size
        self.is_val = is_val
        
        # AST specific constants
        self.target_h = 128   # Freq
        self.target_w = 1024  # Time (Standard for AST)
        
        # Augmentation
        self.freq_mask = T.FrequencyMasking(freq_mask_param=25)
        self.time_mask = T.TimeMasking(time_mask_param=50)

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        # Balanced sampling
        genre = self.genres[idx % len(self.genres)]
        songs = self.song_dict[genre]
        
        # 1. Stem Mixing (Power Domain)
        mixed_mel = np.zeros((128, 1292), dtype=np.float32)
        for stem in ["drums.npy", "bass.npy", "vocals.npy", "other.npy"]:
            song_id = random.choice(songs)
            path = os.path.join(self.processed_dir, genre, song_id, stem)
            
            mel = np.load(path)
            # Match width
            if mel.shape[1] > 1292: mel = mel[:, :1292]
            else: mel = np.pad(mel, ((0,0), (0, 1292 - mel.shape[1])))
            
            gain = 1.0 if self.is_val else random.uniform(0.6, 1.4)
            mixed_mel += mel * gain

        # 2. Add ESC-50 Noise
        if not self.is_val and self.noise_files:
            for _ in range(random.randint(1, 3)):
                n_mel = np.load(random.choice(self.noise_files))
                if mixed_mel.shape[1] > n_mel.shape[1]:
                    start = random.randint(0, mixed_mel.shape[1] - n_mel.shape[1])
                    mixed_mel[:, start : start + n_mel.shape[1]] += n_mel * random.uniform(0.05, 0.2)

        # 3. DB Conversion & Normalization
        mel_db = librosa.power_to_db(np.maximum(mixed_mel, 1e-10))
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
        
        # 4. AST Formatting (Transpose to Time x Freq)
        # Input: (128, 1292) -> Output: (1024, 128)
        mel_t = mel_db.T # (1292, 128)
        
        if self.is_val:
            # Centered crop for validation
            start = (mel_t.shape[0] - self.target_w) // 2
            mel_final = mel_t[start : start + self.target_w, :]
        else:
            # Random crop for training (Augmentation!)
            start = random.randint(0, mel_t.shape[0] - self.target_w)
            mel_final = mel_t[start : start + self.target_w, :]

        mel_final = torch.tensor(mel_final, dtype=torch.float32)

        # 5. SpecAugment (Frequency/Time masking)
        if not self.is_val:
            mel_final = mel_final.unsqueeze(0) # (1, 1024, 128)
            mel_final = self.freq_mask(mel_final)
            mel_final = self.time_mask(mel_final)
            mel_final = mel_final.squeeze(0)

        return mel_final, self.genre_to_idx[genre]

In [12]:
# Paths (Update these based on your Kaggle setup)
PROCESSED_DIR = "/kaggle/working/processed_mels"
NOISE_DIR = "/kaggle/working/processed_noise_mels"

# 1. Create Splits
train_songs, val_songs, genres = get_stratified_split(PROCESSED_DIR)

# 2. Create Datasets
# Training size is large (50k) because we use random combinations/crops
train_dataset = ASTRobustDataset(PROCESSED_DIR, NOISE_DIR, train_songs, genres, size=20000)
val_dataset   = ASTRobustDataset(PROCESSED_DIR, NOISE_DIR, val_songs, genres, size=2000, is_val=True)

# 3. Create Loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

print(f"Ready! Training on {len(train_dataset)} mixed samples.")

Genre blues      | Train:  85 | Val: 15
Genre classical  | Train:  85 | Val: 15
Genre country    | Train:  85 | Val: 15
Genre disco      | Train:  85 | Val: 15
Genre hiphop     | Train:  85 | Val: 15
Genre jazz       | Train:  85 | Val: 15
Genre metal      | Train:  85 | Val: 15
Genre pop        | Train:  85 | Val: 15
Genre reggae     | Train:  85 | Val: 15
Genre rock       | Train:  85 | Val: 15
Ready! Training on 20000 mixed samples.


In [13]:
import torch

import torch.nn as nn

from torch.utils.data import DataLoader

from transformers import ASTConfig, ASTForAudioClassification, ASTFeatureExtractor

from sklearn.metrics import f1_score

import torch.nn.functional as F

from torch.cuda.amp import GradScaler, autocast

from tqdm import tqdm



# --- 1. Model Initialization ---

model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"

feature_extractor = ASTFeatureExtractor.from_pretrained(model_name)



config = ASTConfig.from_pretrained(model_name)

config.num_labels = 10

model = ASTForAudioClassification.from_pretrained(

    model_name, 

    config=config, 

    ignore_mismatched_sizes=True

)



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)



# --- 2. Optimizer & Scheduler ---

# Using a slightly higher LR for the classification head and lower for backbone

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=5e-3)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

scaler = GradScaler() # For FP16 Mixed Precision

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)



# --- 3. Training & Validation Functions ---

def train_one_epoch(model, loader, optimizer, criterion, device):

    model.train()

    total_loss = 0

    all_preds = []

    all_labels = []

    

    pbar = tqdm(loader, desc="Training")

    for inputs, labels in pbar:

        inputs, labels = inputs.to(device), labels.to(device)

        

        optimizer.zero_grad()

        

        # Mixed Precision Forward

        with autocast():

            outputs = model(inputs).logits

            loss = criterion(outputs, labels)

        

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())

        all_labels.extend(labels.cpu().numpy())

        

        pbar.set_postfix(loss=loss.item())

        

    avg_loss = total_loss / len(loader)

    f1 = f1_score(all_labels, all_preds, average='macro')

    return avg_loss, f1



def validate(model, loader, criterion, device):

    model.eval()

    total_loss = 0

    all_preds = []

    all_labels = []

    

    with torch.no_grad():

        for inputs, labels in tqdm(loader, desc="Validating"):

            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs).logits

            loss = criterion(outputs, labels)

            

            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())

            all_labels.extend(labels.cpu().numpy())

            

    avg_loss = total_loss / len(loader)

    f1 = f1_score(all_labels, all_preds, average='macro')

    return avg_loss, f1



# --- 4. Main Loop ---

num_epochs = 10

best_val_f1 = 0.0



for epoch in range(num_epochs):

    print(f"\n--- Epoch {epoch+1}/{num_epochs} ---")

    

    train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, device)

    val_loss, val_f1 = validate(model, val_loader, criterion, device)

    

    scheduler.step()

    

    print(f"Train Loss: {train_loss:.4f} | Train Macro F1: {train_f1:.4f}")

    print(f"Val Loss: {val_loss:.4f} | Val Macro F1: {val_f1:.4f}")

    

    # Save the best model

    if val_f1 > best_val_f1:

        best_val_f1 = val_f1

        torch.save(model.state_dict(), "best_ast_model.pt")

        print("⭐ New best model saved!")



print(f"\nTraining Complete. Best Validation Macro F1: {best_val_f1:.4f}")

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                          
------------------------+----------+------------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
/tmp/ipykernel_55/2224638209.py:57: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() # For FP16 Mixed Precision


...

Train Loss: 0.5100 | Train Macro F1: 0.9966
Val Loss: 0.5882 | Val Macro F1: 0.9623
⭐ New best model saved!

--- Epoch 9/10 ---


Training:   0%|          | 0/1250 [00:00<?, ?it/s]/tmp/ipykernel_55/2224638209.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Validating: 100%|██████████| 125/125 [03:03<00:00,  1.47s/it]


Train Loss: 0.5070 | Train Macro F1: 0.9979
Val Loss: 0.5955 | Val Macro F1: 0.9634
⭐ New best model saved!

--- Epoch 10/10 ---


Training:   0%|          | 0/1250 [00:00<?, ?it/s]/tmp/ipykernel_55/2224638209.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Validating: 100%|██████████| 125/125 [03:03<00:00,  1.47s/it]


Train Loss: 0.5059 | Train Macro F1: 0.9984
Val Loss: 0.5856 | Val Macro F1: 0.9650
⭐ New best model saved!

Training Complete. Best Validation Macro F1: 0.9650


In [15]:
import pandas as pd
import torch
import os
import numpy as np
import librosa
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

class ASTSubmissionDataset(Dataset):
    def __init__(self, test_csv, mashup_dir, target_w=1024):
        self.df = pd.read_csv(test_csv)
        self.mashup_dir = mashup_dir
        self.target_w = target_w # AST standard time frames

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Handle filename column differences
        filename = row["filename"] if "filename" in row else f"{str(row['id']).zfill(4)}.wav"
        path = os.path.join(self.mashup_dir, filename)

        # 1. Load Audio
        y, _ = librosa.load(path, sr=22050, duration=30.0)
        if len(y) < 22050 * 30:
            y = np.pad(y, (0, 22050 * 30 - len(y)))

        # 2. Convert to Mel Spectrogram (Match your training params)
        mel = librosa.feature.melspectrogram(y=y, sr=22050, n_mels=128, n_fft=2048, hop_length=512)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        
        # 3. Normalization (Critical for AST)
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

        # 4. AST Formatting: (Freq, Time) -> (Time, Freq)
        mel_t = mel_db.T # (Time_steps, 128)

        # 5. Precise Resize to 1024
        if mel_t.shape[0] > self.target_w:
            # Centered crop for inference stability
            start = (mel_t.shape[0] - self.target_w) // 2
            mel_final = mel_t[start : start + self.target_w, :]
        else:
            mel_final = np.pad(mel_t, ((0, self.target_w - mel_t.shape[0]), (0, 0)))

        return torch.tensor(mel_final, dtype=torch.float32), str(row["id"])

# --- Settings ---
test_csv = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
mashup_dir = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
genres = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]

test_dataset = ASTSubmissionDataset(test_csv, mashup_dir)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# --- Inference Loop ---
model.eval()
predictions = []

with torch.no_grad():
    for mels, ids in tqdm(test_loader, desc="Inference"):
        mels = mels.to(device) # Shape: [Batch, 1024, 128]
        
        outputs = model(mels)
        # Handle if model returns a dict (HuggingFace AST) or a raw tensor
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs
        
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        
        for sample_id, pred_idx in zip(ids, preds):
            predictions.append({
                "id": sample_id.zfill(4), 
                "genre": genres[pred_idx]
            })

# --- Save Results ---
submission = pd.DataFrame(predictions)
submission.to_csv("submission.csv", index=False)
print(f"Saved {len(submission)} predictions to submission.csv")

Inference: 100%|██████████| 95/95 [04:28<00:00,  2.83s/it]

Saved 3020 predictions to submission.csv


In [16]:
submission

,id,genre
0,0001,pop
1,0002,jazz
2,0003,disco
3,0004,metal
4,0005,country
...,...,...
3015,3016,rock
3016,3017,jazz
3017,3018,reggae
3018,3019,country
